In [2]:
import pandas as pd
import geopandas as gpd


In [3]:
metrodf = pd.read_csv("POWER_Point_Hourly_20260501_20260531_020d28N_085d78E_UTC.csv", skiprows=13, header=0)
nodesdf = pd.read_csv("nodes.csv")
edgesdf = pd.read_csv("edges.csv")

In [4]:
metrodf.head()

,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DWN,ALLSKY_SFC_LW_DWN,T2M,QV2M,RH2M
0,2026,5,1,0,0.25,1.49,26.12,20.24,94.63
1,2026,5,1,1,0.82,1.52,27.20,20.63,90.53
2,2026,5,1,2,1.44,1.56,28.76,20.37,81.70
3,2026,5,1,3,2.36,1.54,30.16,19.61,72.63
4,2026,5,1,4,3.20,1.55,31.72,18.78,63.64


In [5]:
nodesdf.head()

,osmid,y,x,street_count,geometry
0,8065246087,20.274953,85.777867,1,POINT (85.7778668 20.2749529)
1,8065246096,20.274912,85.778567,3,POINT (85.7785672 20.2749124)
2,8065209270,20.276272,85.778456,3,POINT (85.7784557 20.2762723)
3,8065209273,20.276157,85.775806,1,POINT (85.7758059 20.2761566)
4,2997273652,20.275541,85.778512,3,POINT (85.7785119 20.2755415)


In [6]:
edgesdf.head()

,u,v,key,osmid,access,highway,oneway,reversed,length,geometry,junction
0,8065246087,8065246096,0,1569667948,private,service,False,False,75.158115,"LINESTRING (85.7778668 20.2749529, 85.777906 2...",NaN
1,8065246096,14299139685,0,230784863,private,service,False,False,116.225300,"LINESTRING (85.7785672 20.2749124, 85.7786588 ...",NaN
2,8065246096,2997273652,0,230784863,private,service,False,True,70.190233,"LINESTRING (85.7785672 20.2749124, 85.7785119 ...",NaN
3,8065246096,8065246087,0,1569667948,private,service,False,True,75.158115,"LINESTRING (85.7785672 20.2749124, 85.7782859 ...",NaN
4,8065209270,2392366807,0,230784863,private,service,False,False,67.069064,"LINESTRING (85.7784557 20.2762723, 85.7785005 ...",NaN


In [10]:
date = [5, 10, 14]
params = metrodf[(metrodf['MO']==date[0]) & (metrodf['DY']==date[1]) & (metrodf['HR']==date[2])]
params

,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DWN,ALLSKY_SFC_LW_DWN,T2M,QV2M,RH2M
230,2026,5,10,14,0.0,1.49,28.35,16.08,66.35


In [29]:
temp_penalty = max(0, params['T2M'].item()-22) * 0.08
humidity_penalty = params['RH2M'] * 0.15

heat_score = 1 + temp_penalty + humidity_penalty
edgesdf['cost'] = edgesdf['length'] * heat_score.iloc[0]


In [31]:
edgesdf.head()

,u,v,key,osmid,access,highway,oneway,reversed,length,geometry,junction,cost
0,8065246087,8065246096,0,1569667948,private,service,False,False,75.158115,"LINESTRING (85.7778668 20.2749529, 85.777906 2...",NaN,861.349578
1,8065246096,14299139685,0,230784863,private,service,False,False,116.225300,"LINESTRING (85.7785672 20.2749124, 85.7786588 ...",NaN,1332.000047
2,8065246096,2997273652,0,230784863,private,service,False,True,70.190233,"LINESTRING (85.7785672 20.2749124, 85.7785119 ...",NaN,804.415166
3,8065246096,8065246087,0,1569667948,private,service,False,True,75.158115,"LINESTRING (85.7785672 20.2749124, 85.7782859 ...",NaN,861.349578
4,8065209270,2392366807,0,230784863,private,service,False,False,67.069064,"LINESTRING (85.7784557 20.2762723, 85.7785005 ...",NaN,768.645010


In [35]:
import networkx as nx

def dijkstra_route(edgesdf, snode, tnode):
    G = nx.from_pandas_edgelist(
        edgesdf,
        source='u',
        target='v',
        edge_attr=True
    )

    try:
        path_nodes = nx.shortest_path(G, source=snode, target=tnode, weight='cost')
        total_cost = nx.path_weight(G, path_nodes, weight='cost')
        return path_nodes, total_cost

    except nx.NetworkXNoPath:
        print(f"Error: No valid path found between node {start_node} and node {target_node}.")
        return [], float('inf')

    except nx.NodeNotFound as e:
        print(f"Error: Start or target node not found in graph: {e}")
        return [], float('inf')